In [1]:
import numpy as np 
import pandas as pd 
np.random.seed(42)

In [2]:
n = 25000
S0_MIN, S0_MAX = 60, 140
K_MIN, K_MAX = 80, 120
T_MIN, T_MAX = 0.05, 2.00
R_MIN, R_MAX = 0.00, 0.10
SIGMA_MIN, SIGMA_MAX = 0.10, 0.50


In [3]:
S0 = np.random.uniform(S0_MIN, S0_MAX, n)
K = np.random.uniform(K_MIN, K_MAX, n)
T = np.random.uniform(T_MIN, T_MAX, n)
r = np.random.uniform(R_MIN, R_MAX, n)
sigma = np.random.uniform(SIGMA_MIN, SIGMA_MAX, n)

In [6]:
S0 , K , T , sigma , r 

(array([ 89.96320951, 136.05714451, 118.55951534, ..., 133.86192691,
        110.98285175, 136.14708376], shape=(25000,)),
 array([ 83.69483357,  82.43014918, 104.16767803, ..., 109.95980339,
        100.84043647, 114.46826842], shape=(25000,)),
 array([1.70211133, 1.01430824, 0.43115794, ..., 1.9102015 , 1.91966451,
        0.18860135], shape=(25000,)),
 array([0.33231162, 0.31078866, 0.24041478, ..., 0.29887772, 0.48717294,
        0.32218219], shape=(25000,)),
 array([0.00018843, 0.09599844, 0.05353166, ..., 0.06744534, 0.04994472,
        0.03899085], shape=(25000,)))

In [8]:
r.shape

(25000,)

In [9]:
df = pd.DataFrame({
    "S0":S0 ,
    "K":K ,
    "T":T ,
    "r":r , 
    "sigma":sigma
})

In [10]:
df.head()

,S0,K,T,r,sigma
0,89.963210,83.694834,1.702111,0.000188,0.332312
1,136.057145,82.430149,1.014308,0.095998,0.310789
2,118.559515,104.167678,0.431158,0.053532,0.240415
3,107.892679,118.644653,1.486451,0.070040,0.297285
4,72.481491,100.108852,0.866422,0.091260,0.246039


In [11]:
def crr_put_price(
        S0: float ,
        K: float,
        T: float,
        r: float,
        sigma: float,
        steps: int,
        american: bool
        ):
    if S0 <= 0 or K <= 0:
        raise ValueError("SO and K must be positive")
    if T <= 0:
        return max(K -S0 , 0.0)
    if sigma <= 0:
        raise ValueError("sigma must be positive ")
    if int(steps) != steps or steps < 1:
        raise ValueError("steps must be positive inteher")
    
    steps = int(steps)
    dt = T / steps 
    u = math.exp(sigma * math.sqrt(dt))
    d = 1.0 / u 
    growth = math.exp(r*dt)
    p = (growth - d) / (u - d)
    disc = math.exp(-r*dt)

    if not (0.0 < p < 1.0):
        raise ValueError("invalid risk neutral probablity : increase the steps size or check your inputs")
    
    j = np.arange(steps + 1)
    stock = S0 * (u ** j) * (d ** (steps - j))
    value = np.maximum(K - stock , 0.0)

    for i in range(steps - 1 , -1 , -1):
        value = disc * ( p * value[1:i+2] + (1.0 - p) * value[0:i+1])

        if american: 
            j = np.arange(i + 1)
            stock = S0 * (u ** j) * (d ** (i - j))
            exercise = np.maximum(K - stock , 0.0)
            value = np.maximum(value , exercise)

    return float(value[0])       




In [12]:
def generate_option_labels(df,steps=300,american=True):
    df=df.copy()
    df["Put_Price"]=df.apply(lambda row: crr_put_price(S0=row["S0"],K=row["K"],T=row["T"],r=row["r"],sigma=row["sigma"],steps=steps,american=american),axis=1)
    return df

In [17]:
import math

In [19]:
pricing_steps = 300

df = generate_option_labels(
    df,
    steps=pricing_steps,
    american=True
)

df.head()

,S0,K,T,r,sigma,Put_Price
0,89.963210,83.694834,1.702111,0.000188,0.332312,11.968496
1,136.057145,82.430149,1.014308,0.095998,0.310789,0.354892
2,118.559515,104.167678,0.431158,0.053532,0.240415,1.579463
3,107.892679,118.644653,1.486451,0.070040,0.297285,17.218931
4,72.481491,100.108852,0.866422,0.091260,0.246039,27.627361


In [22]:
# sanity check : the put price must be greater than 0 
df[df['Put_Price'] < 0 ].sum()

S0           0.0
K            0.0
T            0.0
r            0.0
sigma        0.0
Put_Price    0.0
dtype: float64

In [24]:
intrinsic=np.maximum(df["K"]-df["S0"],0)

print("Below Intrinsic :", (df["Put_Price"]<intrinsic).sum())
print("Infinite Values :", np.isinf(df["Put_Price"]).sum())
print("NaN Values      :", df["Put_Price"].isna().sum())

Below Intrinsic : 0
Infinite Values : 0
NaN Values      : 0


In [25]:
df.to_csv("synthetic_american_put_dataset.csv",index=False)

print("Dataset Saved Successfully.")
print(df.shape)

Dataset Saved Successfully.
(25000, 6)
